# Phase 0 (PVSG) — Data Exploration

**Goal of this notebook:** the same rigor applied to the ASPIRe exploration
notebook, now applied to PVSG (Panoptic Video Scene Graph Generation,
Yang et al., CVPR'23) — parse, cross-reference, and visually verify before
trusting anything, and produce a like-for-like comparison against ASPIRe.

## Critical differences from ASPIRe you should know before using this

1. **Video sources do not overlap with TAO-Amodal/ASPIRe at all.** PVSG
   uses VidOR (289 videos), EPIC-Kitchens (55), and Ego4D (56) — a totally
   separate download, not reusable from your existing `frames/` folder.
2. **PVSG has ONE relation type**, not five. ASPIRe/HIG's Appearance /
   Situation / Position / Interaction / Relation split does not exist here
   — there is a single flat list of 57 relation predicates covering
   spatial, action, and human-action relations together. This means PVSG
   cannot be used to train HIG's actual hierarchical multi-attribute
   architecture in the way the paper describes it -- at best it supports a
   single-task relation-classification comparison point.
3. **Objects are grounded by panoptic segmentation masks, not bounding
   boxes.** The `objects` list per video carries no per-frame timing at
   all -- object presence in a given frame has to be read from the mask
   files themselves. Only `relations` carry explicit frame ranges.
4. PVSG adds two things ASPIRe doesn't have: free-text **captions** (per
   time segment) and **QA pairs** -- both could be useful signal later,
   independent of the relation-graph task.

This notebook first works entirely from `pvsg.json` (3.88MB, contains every
annotation -- objects, relations, captions, QA, summaries) since that alone
lets us do most of the same checks as the ASPIRe notebook, without needing
the much larger mask/video downloads (10.8GB total). The mask/frame-level
visual sanity check section is included but will gracefully report
"not found" until you've downloaded and extracted at least one source's
zip files.


In [1]:
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

%matplotlib inline


## 0.1 Paths

`pvsg.json` (3.88MB) can be downloaded directly from
https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json --
no need to clone the whole repo or download the 10.8GB of masks/videos just
for annotation-level exploration.

Mirroring OpenPVSG's own expected structure (so any of their scripts still
work unmodified if you use them later), assumed layout:

```
master-degree-project/
└── data/PVSG_Dataset/
    ├── pvsg.json
    └── data/
        ├── vidor/{frames, masks, videos}
        ├── epic_kitchen/{frames, masks, videos}
        └── ego4d/{frames, masks, videos}
```

Adjust `PROJECT_ROOT` below if this notebook doesn't sit at
`src/notebooks/`, same as the ASPIRe notebook.


In [3]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

PVSG_ROOT = PROJECT_ROOT / "data" / "PVSG"
PVSG_JSON = PVSG_ROOT / "pvsg.json"
PVSG_DATA_ROOT = PVSG_ROOT / "data"  # frames/masks/videos per source live under here

print("PROJECT_ROOT   :", PROJECT_ROOT, "-> exists:", PROJECT_ROOT.exists())
print("PVSG_ROOT      :", PVSG_ROOT, "-> exists:", PVSG_ROOT.exists())
print("PVSG_JSON      :", PVSG_JSON, "-> exists:", PVSG_JSON.exists())
print("PVSG_DATA_ROOT :", PVSG_DATA_ROOT, "-> exists:", PVSG_DATA_ROOT.exists())

assert PVSG_JSON.exists(), (
    "pvsg.json not found -- download it from "
    "https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json "
    "and place it at the PVSG_JSON path above before continuing."
)


PROJECT_ROOT   : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project -> exists: True
PVSG_ROOT      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\PVSG -> exists: True
PVSG_JSON      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\PVSG\pvsg.json -> exists: True
PVSG_DATA_ROOT : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\PVSG\data -> exists: True


## 0.2 Load and verify top-level structure

Confirmed real schema of `pvsg.json` (verified directly, not from the
paper's prose):

```
pvsg.json
├── objects: {thing: [...], stuff: [...]}   # 115 thing + 11 stuff = 126 total
├── relations: [...]                         # 57 strings, single flat list
├── split: {vidor: {train:[...], val:[...]}, epic_kitchen: {...}, ego4d: {...}}
└── data: [ { video_id, meta, objects, relations, captions, qa_pairs, summary }, ... ]
```

The cell below cross-checks vocabulary sizes and video counts against the
paper's own published numbers (126 object classes, 57 relation classes,
289+55+56=400 videos) -- the same integrity check we ran for ASPIRe against
its project page.


In [4]:
with open(PVSG_JSON, "r") as f:
    pvsg = json.load(f)

print("Top-level keys:", list(pvsg.keys()))
print()

n_thing = len(pvsg["objects"]["thing"])
n_stuff = len(pvsg["objects"]["stuff"])
n_relations = len(pvsg["relations"])

print(f"thing categories: {n_thing}")
print(f"stuff categories: {n_stuff}")
print(f"thing+stuff total: {n_thing + n_stuff}  (paper claims 126)")
print(f"relation predicates: {n_relations}  (paper claims 57)")
print()

split_counts = {src: {sp: len(ids) for sp, ids in splits.items()} for src, splits in pvsg["split"].items()}
total_videos = sum(sum(sp.values()) for sp in split_counts.values())
print("Videos per source/split:", split_counts)
print(f"Total videos: {total_videos}  (paper claims 400: 289 VidOR + 55 EPIC-Kitchens + 56 Ego4D)")
print()
print(f"Entries in 'data': {len(pvsg['data'])}")


Top-level keys: ['objects', 'relations', 'split', 'data']

thing categories: 115
stuff categories: 11
thing+stuff total: 126  (paper claims 126)
relation predicates: 57  (paper claims 57)

Videos per source/split: {'vidor': {'train': 249, 'val': 40}, 'epic_kitchen': {'train': 44, 'val': 11}, 'ego4d': {'train': 45, 'val': 11}}
Total videos: 400  (paper claims 400: 289 VidOR + 55 EPIC-Kitchens + 56 Ego4D)

Entries in 'data': 400


## 0.3 Side-by-side comparison with ASPIRe

| | ASPIRe | PVSG |
|---|---|---|
| Videos | 1,488 (500 train / 988 val) | 400 (338 train / 62 val across 3 sources) |
| Video sources | ArgoVerse, BDD, Charades, HACS, LaSOT, YFCC100M, AVA | VidOR, EPIC-Kitchens, Ego4D |
| Overlap with TAO-Amodal frames | Full (same source videos) | **None** |
| Object grounding | Bounding boxes | Panoptic segmentation masks |
| Predicate structure | 5 separate types (Appearance/Situation/Position/Interaction/Relation) | 1 flat relation type (57 predicates) |
| Predicate vocabulary size | 4,549 total across 5 types (722/2902/130/565/230) | 57 |
| Extra signal | none | free-text captions + QA pairs per video |
| Confidence/score field | none | none |
| Reference implementation | `aspire.py`, broken against public JSON, only reads one field | Full `OpenPVSG` codebase with working baselines (IPS+T, VPS stages) |

The last row matters a lot: **PVSG actually has a working, documented
reference implementation** (unlike ASPIRe) -- Table 2 in the paper reports
real R@K/mR@K numbers for multiple baselines. This makes PVSG considerably
lower-risk from an engineering-verification standpoint, even though it
only supports a single-task relation model rather than HIG's five-way
hierarchy.


## 0.4 Coverage check -- which videos have frames/masks/videos on disk

Same logic as the ASPIRe notebook's section 0.4: report exactly what
fraction of each source's videos are available locally, broken out by
split. Expect this to show 0% until you've downloaded and extracted at
least one source's zip files from
https://huggingface.co/datasets/Jingkang/PVSG/tree/main
(`VidOR/`, `EpicKitchen/`, `Ego4D/` -- each has a `_masks.zip` and
`_videos.zip`; extract per OpenPVSG's `unzip_and_extract.py` so you end up
with `data/<source>/{frames,masks,videos}/<video_id>/...`).


In [5]:
SOURCE_DIR_NAMES = {
    "vidor": "vidor",
    "epic_kitchen": "epic_kitchen",
    "ego4d": "ego4d",
}

def check_pvsg_coverage(pvsg_dict, data_root: Path):
    results = []
    for source, splits in pvsg_dict["split"].items():
        source_dir = data_root / SOURCE_DIR_NAMES.get(source, source)
        for split_name, video_ids in splits.items():
            for vid in video_ids:
                frames_dir = source_dir / "frames" / vid
                masks_dir = source_dir / "masks" / vid
                results.append({
                    "source": source,
                    "split": split_name,
                    "video_id": vid,
                    "frames_exist": frames_dir.exists(),
                    "masks_exist": masks_dir.exists(),
                })
    return results


coverage = check_pvsg_coverage(pvsg, PVSG_DATA_ROOT)

by_source = defaultdict(lambda: {"total": 0, "frames": 0, "masks": 0})
for r in coverage:
    by_source[r["source"]]["total"] += 1
    by_source[r["source"]]["frames"] += int(r["frames_exist"])
    by_source[r["source"]]["masks"] += int(r["masks_exist"])

print("Coverage by source (frames / masks present locally, out of total annotated videos):")
for source, counts in by_source.items():
    total = counts["total"]
    print(f"  {source:14s}: frames {counts['frames']:3d}/{total:3d}  ({counts['frames']/total:.0%})   "
          f"masks {counts['masks']:3d}/{total:3d}  ({counts['masks']/total:.0%})")


Coverage by source (frames / masks present locally, out of total annotated videos):
  vidor         : frames   0/289  (0%)   masks   0/289  (0%)
  epic_kitchen  : frames   0/ 55  (0%)   masks   0/ 55  (0%)
  ego4d         : frames   0/ 56  (0%)   masks   0/ 56  (0%)


## 0.5 Deep dive on a single video entry

Decode every field of one `data` entry: static object list, relation
triplets (with frame ranges converted to seconds via `fps`), captions, QA
pairs, and the overall summary. This is the PVSG equivalent of the ASPIRe
notebook's section 0.5.


In [6]:
def seconds(frame_idx, fps):
    return round(frame_idx / fps, 2)


sample_video = pvsg["data"][0]
meta = sample_video["meta"]
fps = meta["fps"]

print(f"video_id: {sample_video['video_id']}")
print(f"meta: {meta}")
print()

print("=== Objects (static per-video list; no per-frame timing here) ===")
obj_lookup = {}
for obj in sample_video["objects"]:
    obj_lookup[obj["object_id"]] = obj
    kind = "thing" if obj["is_thing"] else "stuff"
    print(f"  object_id={obj['object_id']:3d}  category='{obj['category']}'  ({kind})"
          + (f"  status={obj['status']}" if obj["status"] else ""))

print()
print("=== Relations (subject_id, object_id, predicate, [ [start,end] frame ranges ]) ===")
for subj_id, obj_id, predicate, spans in sample_video["relations"]:
    subj_cat = obj_lookup.get(subj_id, {}).get("category", "?")
    obj_cat = obj_lookup.get(obj_id, {}).get("category", "?")
    spans_sec = [f"{seconds(s, fps)}s-{seconds(e, fps)}s" for s, e in spans]
    print(f"  [{subj_id}:{subj_cat}] --[{predicate}]--> [{obj_id}:{obj_cat}]   spans: {spans_sec}")

print()
print("=== Captions ===")
for cap in sample_video.get("captions", []):
    print(f"  {cap['time']}: {cap['description']}")

print()
print("=== QA pairs ===")
for qa in sample_video.get("qa_pairs", []):
    print(f"  [{qa['time']}] Q: {qa['question']}")
    print(f"           A: {qa['answer']}")

print()
print("=== Summary ===")
print(" ", sample_video.get("summary", ""))


video_id: 0001_4164158586
meta: {'height': 360, 'width': 480, 'fps': 5, 'duration': 36.0, 'num_frames': 180}

=== Objects (static per-video list; no per-frame timing here) ===
  object_id=  1  category='wall'  (stuff)
  object_id=  2  category='countertop'  (thing)
  object_id=  3  category='microwave'  (thing)
  object_id=  4  category='cabinet'  (thing)
  object_id=  5  category='door'  (thing)
  object_id=  6  category='fridge'  (thing)
  object_id=  7  category='adult'  (thing)
  object_id=  8  category='child'  (thing)
  object_id=  9  category='table'  (thing)
  object_id= 10  category='chair'  (thing)
  object_id= 11  category='candle'  (thing)
  object_id= 12  category='cake'  (thing)
  object_id= 13  category='countertop'  (thing)
  object_id= 14  category='cabinet'  (thing)
  object_id= 15  category='door'  (thing)
  object_id= 16  category='adult'  (thing)
  object_id= 17  category='child'  (thing)
  object_id= 18  category='baby'  (thing)
  object_id= 19  category='table'  

## 0.6 Relation and object category frequency (long-tail check)

Same kind of check as the ASPIRe notebook's section 0.8 -- confirm how
long-tailed PVSG's single relation vocabulary is, and check object category
frequency too. Unlike ASPIRe's open-vocabulary free text, PVSG's 57
relations are a small closed set, so we can show the *entire* distribution,
not just a top-N.


In [7]:
relation_counts = Counter()
object_category_counts = Counter()

for entry in pvsg["data"]:
    for subj_id, obj_id, predicate, spans in entry["relations"]:
        relation_counts[predicate] += 1
    for obj in entry["objects"]:
        object_category_counts[obj["category"]] += 1

print(f"Distinct relations used: {len(relation_counts)} / {n_relations} in vocabulary")
print()
print("Full relation frequency (most to least common):")
for pred, cnt in relation_counts.most_common():
    print(f"  {cnt:5d}  {pred}")


Distinct relations used: 65 / 57 in vocabulary

Full relation frequency (most to least common):
    918  holding
    836  on
    250  in front of
    214  sitting on
    212  standing on
    194  touching
    153  walking on
    149  looking at
    147  opening
    129  beside
    105  playing with
     99  closing
     83  running on
     76  picking
     67  in
     65  next to
     62  blowing
     60  lying on
     59  throwing
     49  cleaning
     41  talking to
     38  wearing
     37  caressing
     36  kissing
     29  carrying
     27  hugging
     27  brushing
     25  kicking
     24  cutting
     21  pushing
     21  chasing
     21  hitting
     19  playing
     19  biting
     18  swinging
     18  stirring
     17  pulling
     16  eating
     16  hanging from
     15  catching
     15  toward
     14  pointing to
     11  squatting on
     11  shaking hand with
     11  watering
     10  over
      9  jumping from
      9  riding
      9  entering
      8  guiding
  

In [8]:
print(f"Distinct object categories used: {len(object_category_counts)} / {n_thing + n_stuff} in vocabulary")
print()
print("Top 20 object categories:")
for cat, cnt in object_category_counts.most_common(20):
    print(f"  {cnt:5d}  {cat}")


Distinct object categories used: 126 / 126 in vocabulary

Top 20 object categories:
    859  adult
    349  chair
    314  cabinet
    306  child
    294  wall
    258  floor
    215  table
    174  others
    164  toy
    160  plate
    152  door
    149  bottle
    141  ball
    136  sofa
    129  window
    124  cup
    120  box
    118  bowl
     96  countertop
     92  dog


## 0.7 Status field check

Every `status` field shown above was empty (`[]`). Confirm whether that
holds across the entire dataset, or whether it's populated for some
objects/videos (it may encode things like "open/closed" states the paper
doesn't detail in the main text).


In [9]:
non_empty_status = 0
total_objects = 0
for entry in pvsg["data"]:
    for obj in entry["objects"]:
        total_objects += 1
        if obj["status"]:
            non_empty_status += 1
            if non_empty_status <= 5:
                print("Example non-empty status:", obj)

print(f"\nObjects with non-empty 'status': {non_empty_status} / {total_objects}")



Objects with non-empty 'status': 0 / 7596


## 0.8 Visual sanity check -- panoptic mask overlay

**Important caveat:** the mask pixel-value encoding below is a documented
*hypothesis*, not a confirmed fact -- unlike the ASPIRe notebook, we could
not directly inspect a real mask file while building this (the mask zips
are large, gated behind the HF dataset download, and not practical to
fetch just to peek inside). The most common convention for this kind of
VOS/AOT-based annotation pipeline (as described in the paper's Figure 4) is
a single-channel index PNG where **pixel value == object_id**. Verify this
visually the first time you run this cell with real data -- if the overlay
doesn't line up with real object boundaries, the encoding is different
(e.g. RGB-encoded, or a separate id-mapping file) and this function will
need adjusting.


In [10]:
def find_first_available_video_with_masks(pvsg_dict, data_root: Path):
    for source, splits in pvsg_dict["split"].items():
        source_dir = data_root / SOURCE_DIR_NAMES.get(source, source)
        for split_name, video_ids in splits.items():
            for vid in video_ids:
                frames_dir = source_dir / "frames" / vid
                masks_dir = source_dir / "masks" / vid
                if frames_dir.exists() and masks_dir.exists():
                    frame_files = sorted(frames_dir.glob("*.png")) + sorted(frames_dir.glob("*.jpg"))
                    mask_files = sorted(masks_dir.glob("*.png"))
                    if frame_files and mask_files:
                        return vid, source, frame_files[0], mask_files[0]
    return None, None, None, None


vid, source, frame_path, mask_path = find_first_available_video_with_masks(pvsg, PVSG_DATA_ROOT)

if vid is None:
    print("No local frame+mask pair found yet under", PVSG_DATA_ROOT)
    print("-> download and extract at least one source (e.g. VidOR) to run this check.")
else:
    print(f"Using video_id={vid} ({source})")
    print(f"  frame: {frame_path}")
    print(f"  mask : {mask_path}")

    img = Image.open(frame_path).convert("RGB")
    mask = Image.open(mask_path)
    mask_arr = np.array(mask)

    # Hypothesis: single-channel index mask where pixel value == object_id
    if mask_arr.ndim == 3:
        print("\nNOTE: mask has multiple channels -- the single-channel "
              "object_id hypothesis does NOT hold as-is. Inspect mask_arr "
              "directly (e.g. np.unique per channel) before trusting the "
              "overlay below.")
        mask_ids = mask_arr[..., 0]  # best-effort fallback, verify visually
    else:
        mask_ids = mask_arr

    unique_ids = [i for i in np.unique(mask_ids) if i != 0]
    print(f"\nDistinct non-zero mask values in this frame: {unique_ids}")

    video_entry = next(e for e in pvsg["data"] if e["video_id"] == vid)
    obj_lookup = {o["object_id"]: o for o in video_entry["objects"]}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img)
    axes[0].set_title(f"{vid} -- frame")
    axes[0].axis("off")

    axes[1].imshow(img)
    axes[1].imshow(mask_ids, alpha=0.5, cmap="tab20")
    axes[1].set_title("frame + mask overlay (object_id hypothesis)")
    axes[1].axis("off")
    plt.show()

    print("\nDecoded categories for mask values present in this frame:")
    for oid in unique_ids:
        obj = obj_lookup.get(int(oid))
        if obj:
            print(f"  mask value {oid} -> object_id {obj['object_id']}: '{obj['category']}'")
        else:
            print(f"  mask value {oid} -> no matching object_id in this video's object list (hypothesis may be wrong)")


No local frame+mask pair found yet under C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\PVSG\data
-> download and extract at least one source (e.g. VidOR) to run this check.


**Stop and look at the overlay above** (once you have real data to run
this against). Ask yourself:
- Do the colored regions actually align with distinct objects in the frame?
- Do the decoded categories plausibly match what's under each colored
  region?

If not, the pixel-value-equals-object_id hypothesis is wrong for this
mask format, and you'll need to check OpenPVSG's own
`./notebooks/Visualize_Dataset.ipynb` (referenced in their README) for the
authoritative decoding logic before building anything further on top of
masks.


## 0.9 Summary -- what to do before Phase 1

- [ ] Downloaded `pvsg.json` and confirmed counts match paper (126 objects, 57 relations, 400 videos): ______
- [ ] Decided which source(s) to download first (VidOR is largest at 289 videos but also largest zip; EPIC-Kitchens/Ego4D are smaller egocentric sets): ______
- [ ] Confirmed mask pixel-value hypothesis visually once real data is available: ______
- [ ] Compared relation long-tail severity against ASPIRe's (PVSG's 57-class closed vocabulary vs ASPIRe's thousands of open-vocabulary strings) to decide which is more practical as a supervision target: ______
- [ ] Decided how to reconcile PVSG's single relation type with HIG's five-attribute hierarchical design (train a simplified single-task version on PVSG? Use PVSG only as a secondary/comparison signal? Keep ASPIRe as primary?): ______

This notebook does not train anything -- once these are resolved, the next
step mirrors ASPIRe's Phase 1: node feature extraction (from panoptic mask
crops here, instead of bbox crops), k-NN or full graph construction, and a
GAT/TransformerConv relation classifier -- adapted for PVSG's single-task
setup, run alongside the ASPIRe five-attribute pipeline for direct
comparison.
